# ETLegacy Core Infrastructure

Generate a small Einstein Toolkit-style thorn and inspect its CCL and source files.

Navigation: [Index](../index.ipynb) | Previous: [BHaH Project Anatomy](bhah_project_anatomy.ipynb) | Next: [Carpet WaveToy Thorns](carpet_wavetoy_thorns.ipynb)


## Generate a Small Thorn
CCL files declare a thorn interface, parameters, schedule, and source list. The source function here is a compact right-hand-side example.

## Import File and Output Helpers

These standard-library tools run commands, manage temporary project directories, and clean command output.


In [ ]:
from pathlib import Path
import contextlib
import io
import re
import tempfile


## Import ETLegacy Registration Tools

These imports expose the NRPy registries and infrastructure writers used below.


In [ ]:
import nrpy.c_function as cfc
import nrpy.grid as gri
import nrpy.params as par
from nrpy.infrastructures.ETLegacy import (
    CodeParameters,
    MoL_registration,
    Symmetry_registration,
    boundary_conditions,
    interface_ccl,
    make_code_defn,
    param_ccl,
    schedule_ccl,
    simple_loop,
    zero_rhss,
)


## Create a Thorn Workspace

The workspace keeps generated files separate from the tutorial source tree.


In [ ]:
workspace_manager = tempfile.TemporaryDirectory(
    prefix="nrpy_tutorial_etlegacy_", dir=Path.cwd()
)
WORKSPACE = Path(workspace_manager.name)
PROJECT_DIR = WORKSPACE / "project" / "toy_etlegacy"
THORN_NAME = "ToyETLegacyNRPy"


## Step 4: Register gridfunctions

The registry records named grid fields and their roles in generated code.

In [ ]:
for name in [
    f"{THORN_NAME}_rhs_eval",
    f"{THORN_NAME}_zero_rhss",
    f"{THORN_NAME}_MoL_registration",
    f"{THORN_NAME}_Symmetry_registration_oldCartGrid3D",
    f"{THORN_NAME}_specify_Driver_BoundaryConditions",
]:
    cfc.CFunction_dict.pop(name, None)
for name in ["psi", "energy", "psi_rhs", "energy_rhs"]:
    gri.glb_gridfcs_dict.pop(name, None)
par.set_parval_from_str("Infrastructure", "ETLegacy")
par.register_CodeParameter("CCTK_INT", __name__, "fd_order", 4)
par.register_CodeParameter("CCTK_REAL", __name__, "wave_speed", 1.0)
gri.register_gridfunctions(["psi"], group="EVOL")
gri.register_gridfunctions(["energy"], group="AUX")


## Step 5: Build the RHS function body

This body combines ETLegacy parameter reads with a loop over grid points.

In [ ]:
body = f"DECLARE_CCTK_ARGUMENTS_{THORN_NAME}_rhs_eval;\nDECLARE_CCTK_PARAMETERS;\n"
body += CodeParameters.read_CodeParameters(
    list_of_tuples__thorn_CodeParameter=[(THORN_NAME, "wave_speed")],
    declare_invdxxs=True,
)
body += simple_loop.simple_loop(
    loop_body=(
        f"{gri.ETLegacyGridFunction.access_gf('psi_rhs')} = "
        f"wave_speed * ({gri.ETLegacyGridFunction.access_gf('energy')} - "
        f"{gri.ETLegacyGridFunction.access_gf('psi')});"
    ),
    loop_region="interior",
)


## Step 6: Register the RHS C function and its schedule entry

Register the RHS C function and its schedule entry.

In [ ]:
cfc.register_CFunction(
    subdirectory=THORN_NAME,
    includes=["cctk.h", "cctk_Arguments.h", "cctk_Parameters.h"],
    desc="Evaluate a toy right-hand side.",
    cfunc_type="void",
    name=f"{THORN_NAME}_rhs_eval",
    params="CCTK_ARGUMENTS",
    body=body,
    ET_thorn_name=THORN_NAME,
    ET_schedule_bins_entries=[
        (
            "MoL_CalcRHS",
            """
schedule FUNC_NAME in MoL_CalcRHS
{
  LANG: C
  READS: evol_variables(everywhere)
  READS: aux_variables(everywhere)
  WRITES: evol_variables_rhs(interior)
} "Evaluate toy RHS"
""",
        )
    ],
    ET_current_thorn_CodeParams_used=["wave_speed"],
)
print(cfc.CFunction_dict[f"{THORN_NAME}_rhs_eval"].function_prototype)


## Step 7: Register zero-RHS, MoL, symmetry, and boundary hooks

Register zero-RHS, MoL, symmetry, and boundary hooks.

In [ ]:
zero_rhss.register_CFunction_zero_rhss(THORN_NAME)
MoL_registration.register_CFunction_MoL_registration(THORN_NAME)
Symmetry_registration.register_CFunction_Symmetry_registration_oldCartGrid3D(THORN_NAME)
boundary_conditions.register_CFunction_specify_Driver_BoundaryConditions(
    thorn_name=THORN_NAME
)
print("registered C functions:")
for name in sorted(cfc.CFunction_dict):
    print(name)


## Step 8: List registered C functions

The registry now contains the RHS and infrastructure hooks that will be written to source files.

In [ ]:
print("registered C functions:")
for name in sorted(cfc.CFunction_dict):
    print(name)


## Step 9: Write ETLegacy thorn files

The constructors write the CCL declarations and generated source files for the thorn.

In [ ]:
generation_output = io.StringIO()
with contextlib.redirect_stdout(generation_output):
    interface_ccl.construct_interface_ccl(
        project_dir=str(PROJECT_DIR),
        thorn_name=THORN_NAME,
        inherits="Boundary Grid MethodofLines",
        USES_INCLUDEs="USES INCLUDE: Symmetry.h\nUSES INCLUDE: Boundary.h\n",
        is_evol_thorn=True,
    )
    param_ccl.construct_param_ccl(project_dir=str(PROJECT_DIR), thorn_name=THORN_NAME)
    schedule_ccl.construct_schedule_ccl(
        project_dir=str(PROJECT_DIR),
        thorn_name=THORN_NAME,
        STORAGE="""
STORAGE: evol_variables[3]
STORAGE: evol_variables_rhs[1]
STORAGE: aux_variables[1]
""",
    )
    make_code_defn.output_CFunctions_and_construct_make_code_defn(
        project_dir=str(PROJECT_DIR), thorn_name=THORN_NAME
    )


## Step 10: Read generator messages

The cleaned messages confirm which files the infrastructure writer produced.

In [ ]:
cleaned = re.sub(r"\x1b\[[0-?]*[ -/]*[@-~]", "", generation_output.getvalue())
cleaned = cleaned.replace(str(WORKSPACE), "<workspace>").replace(
    str(PROJECT_DIR), "project/toy_etlegacy"
)
print("generation output:")
for line in cleaned.splitlines():
    if line.strip():
        print(line.rstrip())


## Step 11: Check required files

The guard stops the notebook with a clear missing-file error if generation did not produce the expected artifacts.

In [ ]:
required_files = [
    "interface.ccl",
    "param.ccl",
    "schedule.ccl",
    "src/make.code.defn",
    f"src/{THORN_NAME}_rhs_eval.c",
]
for relative_path in required_files:
    path = PROJECT_DIR / THORN_NAME / relative_path
    if not path.is_file():
        raise FileNotFoundError(path)


## Step 12: Inspect the generated inventory

The inventory identifies the generated files relevant to this lesson.

In [ ]:
print("thorn inventory:")
for path in sorted((PROJECT_DIR / THORN_NAME).rglob("*")):
    if path.is_file():
        print(path.relative_to(PROJECT_DIR / THORN_NAME))


## Step 13: Read `interface.ccl`

The interface file declares grid variables and inherited capabilities.

In [ ]:
print(
    (PROJECT_DIR / THORN_NAME / "interface.ccl").read_text(
        encoding="utf-8", errors="replace"
    )
)


## Step 14: Read `param.ccl`

The parameter file declares runtime parameters exposed by the thorn.

In [ ]:
print(
    (PROJECT_DIR / THORN_NAME / "param.ccl").read_text(
        encoding="utf-8", errors="replace"
    )
)


## Step 15: Read `schedule.ccl`

The schedule file places functions in Einstein Toolkit schedule bins.

In [ ]:
print(
    (PROJECT_DIR / THORN_NAME / "schedule.ccl").read_text(
        encoding="utf-8", errors="replace"
    )
)


## Step 16: Read `make.code.defn`

This file lists source files compiled into the thorn.

In [ ]:
print(
    (PROJECT_DIR / THORN_NAME / "src" / "make.code.defn").read_text(
        encoding="utf-8", errors="replace"
    )
)


## Step 17: Read the RHS Source

The source excerpt is the generated C function registered earlier.

In [ ]:
print(
    (PROJECT_DIR / THORN_NAME / "src" / f"{THORN_NAME}_rhs_eval.c").read_text(
        encoding="utf-8", errors="replace"
    )
)


The printed CCL and source files show how the same registered C-function idea is expressed as Einstein Toolkit thorn metadata. The schedule entries state when each generated function runs.


## Continue to Carpet WaveToy Thorns
- [C Function Registration](../1-intro/c_function.ipynb)
- [BHaH Project Anatomy](bhah_project_anatomy.ipynb)
- [Carpet WaveToy Thorns](carpet_wavetoy_thorns.ipynb)
- [Backend Choice Guide](backend_choice_guide.ipynb)
